
# 4D-Humans (HMR 2.0) — STAND 4뷰 Full-RGB → SMPL 회귀 초기값

이 노트북은 **STAND front/back/left/right의 원본 full-frame RGB(배경 포함)** 4장을 입력으로 받아,
4D-Humans/HMR2가 만든 SMPL 회귀 결과를 저장합니다.

이 결과의 역할은 **II-8 초기화 재료**입니다.

```text
4D-Humans 결과:
- betas
- body_pose
- global_orient
- weak-perspective camera (참고용)

로컬 II-8에서 추가:
- 4뷰 beta 평균 → shared beta
- metric depth + K → camera-coordinate translation 재초기화
- 최종 init_params.npz 생성

로컬 II-10:
- shared beta + view별 pose/orient/transl을 4뷰 metric depth에 공동 fitting
```

## 반드시 지킬 입력 원칙

- **배경을 지우지 않은 원본 RGB**를 넣으세요.
- 흰 배경 합성, 사람 실루엣 cutout, 알파 채널 PNG, 과도한 crop 이미지는 사용하지 마세요.
- 사람은 프레임 안에 충분히 크게 보이되, 팔·발·머리 끝이 잘리지 않은 것이 좋습니다.
- 이 노트북은 **STAND 4뷰만** 처리합니다. TPOSE는 별도 실행·별도 폴더로 관리하세요.
- 결과 `pred_cam`은 weak-perspective라 metric translation이 아닙니다. II-10에 그대로 넣지 않습니다.

## 산출물

```text
init_params_raw/
├── STAND_front_..._hmr2_raw.npz    # 각 뷰 1명(primary subject) 원시 회귀 결과
├── STAND_back_..._hmr2_raw.npz
├── STAND_left_..._hmr2_raw.npz
├── STAND_right_..._hmr2_raw.npz
├── hmr2_stand_4view_raw.npz         # 4뷰 묶음
└── hmr2_stand_4view_manifest.json   # 원본 해상도·bbox·검출 정보
```

`*_hmr2_raw.npz`는 **II-8 전 단계의 원시 HMR2 결과**입니다.  
공유 beta와 metric translation은 로컬 `ii8_init.py`에서 만듭니다.


## Cell 1 — GPU 확인

In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "런타임을 GPU로 바꿔주세요 (런타임 → 런타임 유형 변경 → T4 GPU)"


## Cell 2 — 설치 (가장 오래 걸림, ~5–10분)

detectron2는 설치된 torch에 맞춰 소스에서 빌드돼. 실패하면 아래 주석 처리한
대체 명령을 써봐. 설치 끝나고 import 에러 나면 런타임 재시작 후 Cell 4부터 진행.

In [ ]:
# 4D-Humans 클론 + 설치
!git clone --recursive https://github.com/shubham-goel/4D-Humans.git
%cd 4D-Humans
!pip install -e .[all] -q

# detectron2 (사람 검출용). .[all]에 포함 안 되면 아래가 설치함.
!pip install "git+https://github.com/facebookresearch/detectron2.git" -q

# 대체안 (위가 실패할 때만 주석 해제):
# !python -m pip install detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu121/torch2.1/index.html


## Cell 3 — SMPL neutral 모델 업로드

https://smplify.is.tue.mpg.de/ 에서 받은
`basicModel_neutral_lbs_10_207_0_v1.0.0.pkl` 파일을 아래 실행 후 업로드 창에서 선택.

In [ ]:
import os, shutil
from pathlib import Path
from google.colab import files

print("basicModel_neutral_lbs_10_207_0_v1.0.0.pkl 을 업로드하세요...")
up = files.upload()
src = list(up.keys())[0]

# 4D-Humans가 기대하는 두 위치에 모두 배치 (코드 버전마다 찾는 곳이 다름)
os.makedirs("data", exist_ok=True)
shutil.copy(src, "data/basicModel_neutral_lbs_10_207_0_v1.0.0.pkl")

cache_smpl = Path.home()/".cache"/"4DHumans"/"data"/"smpl"
cache_smpl.mkdir(parents=True, exist_ok=True)
shutil.copy(src, cache_smpl/"SMPL_NEUTRAL.pkl")
print("SMPL 배치 완료:", cache_smpl/"SMPL_NEUTRAL.pkl")



## Cell 4 — STAND 4뷰 원본 RGB 업로드

아래 4장의 **배경 포함 full-frame RGB**를 업로드하세요.

```text
STAND_front_*.png / jpg
STAND_back_*.png / jpg
STAND_left_*.png / jpg
STAND_right_*.png / jpg
```

이전처럼 배경을 지우거나 사람만 누끼로 만든 이미지는 넣지 마세요.  
4D-Humans는 일반 RGB 사진의 사람 검출·문맥·텍스처를 전제로 동작하므로, 원본 배경이 있는 이미지가 더 자연스럽고 안정적인 초기값을 주는 편입니다.


In [ ]:

import os, shutil
from pathlib import Path
from google.colab import files

INPUT_DIR = Path("my_images")
if INPUT_DIR.exists():
    shutil.rmtree(INPUT_DIR)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

print("STAND 4뷰 원본 RGB(배경 포함)를 업로드하세요...")
uploaded = files.upload()

valid_exts = {".png", ".jpg", ".jpeg"}
for fn in uploaded:
    src = Path(fn)
    if src.suffix.lower() not in valid_exts:
        print(f"[skip] 이미지 형식이 아님: {fn}")
        continue
    shutil.move(str(src), str(INPUT_DIR / src.name))

images = sorted([p for p in INPUT_DIR.iterdir() if p.suffix.lower() in valid_exts])
if len(images) != 4:
    raise RuntimeError(
        f"STAND 4뷰는 정확히 4장이어야 합니다. 현재 {len(images)}장: {[p.name for p in images]}"
    )

required_view_tokens = {"front", "back", "left", "right"}
names_lower = [p.stem.lower() for p in images]
missing = [v for v in sorted(required_view_tokens) if not any(v in n for n in names_lower)]
if missing:
    print(f"[경고] 파일명에서 방향 토큰을 찾지 못했습니다: {missing}")
    print("권장: STAND_front_..., STAND_back_..., STAND_left_..., STAND_right_...")

print("입력 이미지:")
for p in images:
    print(" -", p.name)



## Cell 5 — 공식 demo.py overlay 검수 (권장)

이 결과는 최종 fitting 결과가 아니라, HMR2가 사람을 어느 위치·방향으로 이해했는지 보는 사전 검수입니다.

확인할 것:
- 4장 모두 같은 사람만 검출되었는지
- mesh가 대략 사람의 몸 위에 올라가는지
- front/back/left/right에서 좌우 미러링이나 90° 방향 오류가 없는지
- 발·머리·팔이 크게 잘린 뷰가 없는지

여기서 좀 어긋나도 II-10이 depth로 보정합니다.  
다만 사람이 아닌 배경 객체에 mesh가 올라가거나, 4뷰 중 한 장에서 검출이 실패하면 먼저 입력을 고쳐야 합니다.


In [ ]:
import os
import glob
import shutil
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# 이전 실패 출력 제거
shutil.rmtree("demo_out", ignore_errors=True)

# PyTorch 2.6+와 구형 4D-Humans/HMR2 checkpoint 호환
# 공식 4D-Humans checkpoint를 쓰는 경우에만 적용
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"

# !python은 별도 프로세스라 환경변수를 명령 앞에도 붙여야 함
!TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD=1 python demo.py \
    --img_folder my_images \
    --out_folder demo_out \
    --batch_size 4 \
    --side_view \
    --full_frame

# 결과 표시
outs = sorted(glob.glob("demo_out/*.png"))[:8]

if not outs:
    raise RuntimeError(
        "demo_out에 출력이 없습니다. "
        "위 로그의 첫 번째 Traceback이 실제 원인입니다."
    )

plt.figure(figsize=(14, 4 * ((len(outs) + 1) // 2)))

for i, p in enumerate(outs):
    plt.subplot((len(outs) + 1) // 2, 2, i + 1)
    plt.imshow(mpimg.imread(p))
    plt.axis("off")
    plt.title(os.path.basename(p), fontsize=8)

plt.tight_layout()
plt.show()


## Cell 6 — primary subject 1명만 골라 HMR2 raw init 저장

이 셀은 이미지마다 사람 후보가 여러 명 검출되어도 **가장 큰 primary subject 1명만** 선택합니다.

저장하는 값:
- `betas`: `[10]`
- `global_orient_aa`: `[3]`
- `body_pose_aa`: `[23, 3]`
- `pose_aa`: `[72]` (`global_orient + body_pose`)
- 회전행렬 버전
- `pred_cam`: weak-perspective 카메라. **metric translation으로 사용 금지**
- `pred_keypoints_3d`: 참고·디버깅용
- 선택 bbox·detector score·원본 RGB 해상도

마지막으로 `hmr2_stand_4view_raw.npz`에 4뷰를 한 번에 묶습니다.

중요:
- 여기서는 **translation을 만들지 않습니다.**
- 로컬 II-8에서 final metric depth와 K를 사용해 `transl_init`을 다시 계산해야 합니다.


In [ ]:

import os, json, shutil
from pathlib import Path

import cv2
import numpy as np
import torch
from scipy.spatial.transform import Rotation as Rot

# PyTorch 2.6+ 환경에서 신뢰된 공식 checkpoint 로드를 위한 호환 패치
if not getattr(torch.load, "_hmr2_weights_only_patched", False):
    _original_torch_load = torch.load
    def _patched_torch_load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _original_torch_load(*args, **kwargs)
    _patched_torch_load._hmr2_weights_only_patched = True
    torch.load = _patched_torch_load

from hmr2.models import load_hmr2, download_models, DEFAULT_CHECKPOINT
from hmr2.utils import recursive_to
from hmr2.datasets.vitdet_dataset import ViTDetDataset
from hmr2.utils.utils_detectron2 import DefaultPredictor_Lazy
import hmr2
from detectron2.config import LazyConfig

device = torch.device("cuda")
assert torch.cuda.is_available(), "GPU 런타임이 필요합니다."

# ---------- HMR2 / detector load ----------
download_models()
model, model_cfg = load_hmr2(DEFAULT_CHECKPOINT)
model = model.to(device).eval()

cfg_path = Path(hmr2.__file__).parent / "configs" / "cascade_mask_rcnn_vitdet_h_75ep.py"
dcfg = LazyConfig.load(str(cfg_path))
dcfg.train.init_checkpoint = (
    "https://dl.fbaipublicfiles.com/detectron2/ViTDet/COCO/"
    "cascade_mask_rcnn_vitdet_h/f328730692/model_final_f05665.pkl"
)
for i in range(3):
    dcfg.model.roi_heads.box_predictors[i].test_score_thresh = 0.25
detector = DefaultPredictor_Lazy(dcfg)

def rotmat_to_axis_angle(rotmats: np.ndarray) -> np.ndarray:
    """(..., 3, 3) rotation matrices -> (..., 3) axis-angle."""
    shape = rotmats.shape[:-2]
    out = Rot.from_matrix(rotmats.reshape(-1, 3, 3)).as_rotvec()
    return out.reshape(*shape, 3).astype(np.float32)

def choose_primary_person(boxes_xyxy: np.ndarray, scores: np.ndarray, image_hw):
    """
    단일 피사체 촬영 전제.
    면적이 큰 사람을 우선하고, 프레임 중심에서 지나치게 멀면 약하게 감점한다.
    반환: 선택 index, diagnostic ranking score
    """
    h, w = image_hw
    wh = np.maximum(boxes_xyxy[:, 2:4] - boxes_xyxy[:, 0:2], 1.0)
    area = wh[:, 0] * wh[:, 1]
    centers = 0.5 * (boxes_xyxy[:, 0:2] + boxes_xyxy[:, 2:4])
    image_center = np.array([w * 0.5, h * 0.5], dtype=np.float32)
    diagonal = float(np.hypot(w, h))
    center_distance = np.linalg.norm(centers - image_center[None], axis=1) / max(diagonal, 1.0)

    # 사람이 가장 큰 단일 피사체인 촬영 조건을 반영한 ranking
    rank = scores * np.sqrt(area) * (1.0 - 0.35 * np.clip(center_distance, 0.0, 1.0))
    idx = int(np.argmax(rank))
    return idx, rank

RAW_DIR = Path("init_params_raw")
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

image_exts = {".png", ".jpg", ".jpeg"}
image_paths = sorted([p for p in INPUT_DIR.iterdir() if p.suffix.lower() in image_exts])
if len(image_paths) != 4:
    raise RuntimeError(f"입력 4뷰 검증 실패: {len(image_paths)}장")

records = []
betas_stack = []
body_pose_stack = []
global_orient_stack = []
pred_cam_stack = []
keypoints_stack = []
view_names = []

for img_path in image_paths:
    img_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise RuntimeError(f"이미지를 읽지 못했습니다: {img_path}")

    h, w = img_bgr.shape[:2]
    det = detector(img_bgr)
    inst = det["instances"].to("cpu")

    is_person = inst.pred_classes.numpy() == 0
    scores_all = inst.scores.numpy()
    valid = is_person & (scores_all >= 0.50)

    boxes = inst.pred_boxes.tensor.numpy()[valid]
    scores = scores_all[valid]

    if len(boxes) == 0:
        raise RuntimeError(f"[검출 실패] {img_path.name}: 사람 detection이 없습니다.")

    primary_idx, rank = choose_primary_person(boxes, scores, (h, w))
    selected_box = boxes[primary_idx:primary_idx + 1]
    selected_score = scores[primary_idx:primary_idx + 1]

    if len(boxes) > 1:
        print(
            f"[주의] {img_path.name}: person 후보 {len(boxes)}명. "
            f"primary bbox index={primary_idx}, detector score={selected_score[0]:.3f} 선택"
        )

    dataset = ViTDetDataset(model_cfg, img_bgr, selected_box)
    loader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

    with torch.inference_mode():
        batch = next(iter(loader))
        batch = recursive_to(batch, device)
        out = model(batch)

    smpl = out["pred_smpl_params"]
    global_orient_rotmat = smpl["global_orient"].detach().cpu().numpy()[0]  # [1,3,3]
    body_pose_rotmat = smpl["body_pose"].detach().cpu().numpy()[0]          # [23,3,3]
    betas = smpl["betas"].detach().cpu().numpy()[0].astype(np.float32)      # [10]
    pred_cam = out["pred_cam"].detach().cpu().numpy()[0].astype(np.float32) # [3]
    keypoints_3d = out["pred_keypoints_3d"].detach().cpu().numpy()[0].astype(np.float32)

    global_orient_aa = rotmat_to_axis_angle(global_orient_rotmat).reshape(3)
    body_pose_aa = rotmat_to_axis_angle(body_pose_rotmat).reshape(23, 3)
    pose_aa = np.concatenate([global_orient_aa, body_pose_aa.reshape(-1)], axis=0).astype(np.float32)

    output_path = RAW_DIR / f"{img_path.stem}_hmr2_raw.npz"
    np.savez_compressed(
        output_path,
        schema_version=np.array("hmr2_stand_raw_v2"),
        frame_name=np.array(img_path.name),
        source_rgb_hw=np.array([h, w], dtype=np.int32),
        selected_bbox_xyxy=selected_box[0].astype(np.float32),
        selected_detector_score=selected_score[0].astype(np.float32),
        detected_person_count=np.array(len(boxes), dtype=np.int32),
        global_orient_aa=global_orient_aa,
        body_pose_aa=body_pose_aa,
        pose_aa=pose_aa,
        global_orient_rotmat=global_orient_rotmat.astype(np.float32),
        body_pose_rotmat=body_pose_rotmat.astype(np.float32),
        betas=betas,
        pred_cam_weakpersp=pred_cam,
        pred_keypoints_3d=keypoints_3d,
    )

    records.append({
        "frame_name": img_path.name,
        "source_rgb_hw": [int(h), int(w)],
        "raw_npz": output_path.name,
        "detected_person_count": int(len(boxes)),
        "selected_bbox_xyxy": [float(x) for x in selected_box[0]],
        "selected_detector_score": float(selected_score[0]),
        "primary_ranking_score": float(rank[primary_idx]),
    })

    view_names.append(img_path.stem)
    betas_stack.append(betas)
    body_pose_stack.append(body_pose_aa)
    global_orient_stack.append(global_orient_aa)
    pred_cam_stack.append(pred_cam)
    keypoints_stack.append(keypoints_3d)

    print(f"[ok] {img_path.name} → {output_path.name}")

# 4뷰 raw bundle: II-8 adapter의 편한 입력
bundle_path = RAW_DIR / "hmr2_stand_4view_raw.npz"
np.savez_compressed(
    bundle_path,
    schema_version=np.array("hmr2_stand_4view_raw_v2"),
    view_names=np.asarray(view_names),
    betas_per_view=np.stack(betas_stack, axis=0),                  # [4,10]
    body_pose_aa_per_view=np.stack(body_pose_stack, axis=0),       # [4,23,3]
    global_orient_aa_per_view=np.stack(global_orient_stack, axis=0), # [4,3]
    pred_cam_weakpersp_per_view=np.stack(pred_cam_stack, axis=0),  # [4,3]
    pred_keypoints_3d_per_view=np.stack(keypoints_stack, axis=0),
)

manifest = {
    "schema_version": "hmr2_stand_raw_v2",
    "purpose": (
        "Raw HMR2 regression output for II-8 initialization. "
        "Do not treat pred_cam_weakpersp as metric translation."
    ),
    "input_policy": "Original full-frame RGB with background; no silhouette-only cutout.",
    "views": records,
    "bundle": bundle_path.name,
    "next_step": (
        "Run local ii8_init.py with this raw bundle + 4 matching final metric depth files + K.json. "
        "It must compute shared beta and metric camera-coordinate translation."
    ),
}
with open(RAW_DIR / "hmr2_stand_4view_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("\n완료")
print(" - raw per-view:", len(records))
print(" - bundle:", bundle_path)
print(" - manifest:", RAW_DIR / "hmr2_stand_4view_manifest.json")



## Cell 7 — 결과 확인 및 다운로드

`betas_per_view`가 완전히 같을 필요는 없습니다.  
4뷰 RGB만으로 회귀했기 때문에 값이 다소 흔들리는 것은 정상입니다.

로컬 II-8에서 할 일:

```text
betas_per_view[4,10]
→ robust mean/median 또는 outlier 제외 평균
→ betas_shared_init[10]

body_pose/global_orient
→ 뷰별 초기값 유지

pred_cam_weakpersp
→ 사용하지 않음
final metric depth + K
→ transl_init[4,3] 재계산
```


In [ ]:

import numpy as np, json
from pathlib import Path
from google.colab import files

bundle = np.load(RAW_DIR / "hmr2_stand_4view_raw.npz", allow_pickle=False)

print("view_names:", bundle["view_names"].tolist())
print("betas_per_view:", bundle["betas_per_view"].shape)
print("body_pose_aa_per_view:", bundle["body_pose_aa_per_view"].shape)
print("global_orient_aa_per_view:", bundle["global_orient_aa_per_view"].shape)
print("pred_cam_weakpersp_per_view:", bundle["pred_cam_weakpersp_per_view"].shape)

betas = bundle["betas_per_view"]
print("\nβ per view:")
for name, beta in zip(bundle["view_names"], betas):
    print(f"  {name}: {np.round(beta, 3)}")

print("\nβ mean (참고용, 아직 최종 shared beta 아님):")
print(np.round(betas.mean(axis=0), 3))

manifest_path = RAW_DIR / "hmr2_stand_4view_manifest.json"
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)
print("\n선택된 primary subject:")
for r in manifest["views"]:
    print(f"  {r['frame_name']}: bbox={np.round(r['selected_bbox_xyxy'], 1)}, score={r['selected_detector_score']:.3f}")

!zip -r hmr2_stand_4view_raw.zip init_params_raw -q
files.download("hmr2_stand_4view_raw.zip")



## Cell 8 — 로컬 II-8 / II-10 연결 메모

이 노트북 출력은 II-10에 바로 넣는 최종 `init_params.npz`가 아닙니다.

```python
# local ii8_init.py 의 개념
raw = np.load("hmr2_stand_4view_raw.npz")

betas_per_view = raw["betas_per_view"]                 # [4, 10]
body_pose_init = raw["body_pose_aa_per_view"]           # [4, 23, 3]
global_orient_init = raw["global_orient_aa_per_view"]   # [4, 3]

# robust mean / outlier control
betas_shared_init = robust_average(betas_per_view)      # [10]

# 여기서 HMR2 pred_cam을 쓰지 말 것
# final metric depth + K를 이용해 각 뷰 transl_init을 camera coordinates에서 계산
transl_init = estimate_translation_from_metric_depth(...)

np.savez(
    "init_params.npz",
    betas_shared_init=betas_shared_init,
    body_pose_init=body_pose_init,
    global_orient_init=global_orient_init,
    transl_init=transl_init,
)
```

그리고 II-10은 이 `init_params.npz`를 시작점으로 STAND 4뷰의 depth에 동시에 맞춥니다.


In [ ]:

# 선택: HMR2 raw mesh를 원한다면 demo.py overlay로 충분합니다.
# 이 노트북의 목적은 mesh export가 아니라 II-8용 raw parameter extraction입니다.
# raw HMR2 mesh는 metric scale/translation이 맞지 않으므로, II-10 검증용 mesh로 사용하지 마세요.
print("No-op: use demo_out overlays for HMR2 visual sanity checks.")
